<a href="https://colab.research.google.com/gist/Did-web/585341506bdf837821d42d3f8192575b/capstone1_jupyter-labs-spacex-data-collection-api-v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import pandas as pd
import numpy as np
import datetime
import os

# Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


In [3]:
# URL of the static dump provided by IBM
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

response = requests.get(static_json_url)
if response.status_code == 200:
    data = response.json()
    df_raw = pd.json_normalize(data)

    # Restrict to Falcon 9 launches (optional depending on your progress in the lab)
    # df_raw = df_raw[df_raw['cores'].map(len) == 1] # Example filter from the lab
    print(f"✅ Raw data loaded. Dataset shape: {df_raw.shape}")
else:
    print("❌ Failed to download the static data.")

✅ Raw data loaded. Dataset shape: (107, 42)


In [4]:
# --- CELL: OFFICIAL PARSING FUNCTIONS ---

def getBoosterVersion(data):
    # Fallback dictionary in case the v4 API is down
    fallback = {
        '5e9d0d95eda69955f709d1eb': 'Falcon 1',
        '5e9d0d95eda69973a809d1ec': 'Falcon 9',
        '5e9d0d95eda69974db09d1ed': 'Falcon Heavy'
    }
    for x in data['rocket']:
        if x:
            try:
                # Official IBM v4 logic
                response = requests.get(f"https://api.spacexdata.com/v4/rockets/{x}").json()
                BoosterVersion.append(response['name'])
            except:
                # "Cloud-native" safeguard: avoids a global crash
                BoosterVersion.append(fallback.get(x, "Unknown Rocket"))

def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            try:
                response = requests.get(f"https://api.spacexdata.com/v4/launchpads/{x}").json()
                Longitude.append(response['longitude'])
                Latitude.append(response['latitude'])
                LaunchSite.append(response['name'])
            except:
                # Fallback if the Launchpads API does not respond
                Longitude.append(None)
                Latitude.append(None)
                LaunchSite.append("Unknown Pad")

def getPayloadData(data):
    # Simplified and safe version of the payload extraction (image_d755e3.png)
    for load in data['payloads']:
        if load:
            try:
                # In the static JSON, 'payloads' is a list of IDs (we take the first ID)
                payload_id = load[0] if isinstance(load, list) else load
                response = requests.get(f"https://api.spacexdata.com/v4/payloads/{payload_id}").json()
                PayloadMass.append(response['mass_kg'])
                Orbit.append(response['orbit'])
            except:
                PayloadMass.append(None)
                Orbit.append(None)

def getCoreData(data):
    # Extract first-stage (core) data (image_d75626.png)
    for core in data['cores']:
        if core and len(core) > 0:
            c = core[0] # First core
            try:
                response = requests.get(f"https://api.spacexdata.com/v4/cores/{c['core']}").json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            except:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)

            # Data taken directly from the 'launches' table
            Outcome.append(str(c['landing_success']) + ' ' + str(c['landing_type']))
            Flights.append(c['flight'])
            GridFins.append(c['gridfins'])
            Reused.append(c['reused'])
            Legs.append(c['legs'])
            LandingPad.append(c['landpad'])
        else:
            # Default fill-in when no core is available
            for lst in [Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad]:
                lst.append(None)

In [5]:
# --- CELL: INITIALIZATION AND EXECUTION SCRIPT ---

# 1. Initialize all the global lists required by the lab
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

# 2. Filter the raw data (crucial step of the IBM lab)
# Convert the raw date into a datetime.date object to apply the filter
df_raw['date'] = pd.to_datetime(df_raw['date_utc']).dt.date

# Filter to exclude data after November 13, 2020
df_filtered = df_raw[df_raw['date'] <= datetime.date(2020, 11, 13)].copy()

# Filter to keep only Falcon 9 launches using the rocket ID
df_filtered = df_filtered[df_filtered['rocket'] == '5e9d0d95eda69973a809d1ec'].copy()

# Filter to keep only single-core launches
df_filtered = df_filtered[df_filtered['cores'].map(len) == 1]

# Drop rows without a valid rocket
df_filtered = df_filtered.dropna(subset=['rocket']).reset_index(drop=True)

print(f"Shape after filtering by date and rocket: {df_filtered.shape} rows.")

# 3. Sequentially run the parsing functions (image_d759e5.png)
print("Parsing in progress (this step makes HTTP calls and may take 1 to 2 minutes)...")
getBoosterVersion(df_filtered)
getLaunchSite(df_filtered)
getPayloadData(df_filtered)
getCoreData(df_filtered)
print("🎉 Extraction completed successfully!")

Shape after filtering by date and rocket: (98, 43) rows.
Parsing in progress (this step makes HTTP calls and may take 1 to 2 minutes)...
🎉 Extraction completed successfully!


In [6]:
# --- CELL: ASSEMBLY AND EXPORT ---

launch_dict = {
    'FlightNumber': list(range(1, len(df_filtered) + 1)),
    'Date': df_filtered['date'].tolist(),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

# Create the final analysis DataFrame
df_final = pd.DataFrame(launch_dict)

# Save cleanly to your local/cloud ds sandbox
df_final.to_csv('dataset_part_1.csv', index=False)
print("💾 Final file saved to the ds sandbox!")
df_final.head()

💾 Final file saved to the ds sandbox!


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None
1,2,2010-12-08,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None
2,3,2012-05-22,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None
3,4,2012-10-08,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None
4,5,2013-03-01,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None


In [7]:
df_final.shape

(98, 17)